In [ ]:
# Colab often starts with cwd outside the repo — put the clone on sys.path before any gradient_ascent import.
import os
import pathlib
import subprocess
import sys

_REPO_URL = "https://github.com/haydenyoungcs/gradient-ascent.git"
_REPO_DIR = pathlib.Path("/content/gradient-ascent")
_github = os.environ.get("GITHUB_TOKEN")
_wandb_key = os.environ.get("WANDB_API_KEY")
try:
    from google.colab import drive, userdata  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    if _github is None:
        _github = userdata.get("GITHUB_TOKEN")
    if _wandb_key is None:
        _wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    pass


def _find_project_root_from_cwd() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return None


_project_root = _find_project_root_from_cwd() or _REPO_DIR
if not _project_root.exists():
    if _github:
        _clone = _REPO_URL.replace("https://", f"https://{_github}@")
        subprocess.run(["git", "clone", _clone], check=True)
    else:
        raise RuntimeError(
            "Repo not found. Clone to /content/gradient-ascent or open from repo root; add GITHUB_TOKEN to clone a private repo."
        )

if not (_project_root / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected pyproject.toml under {_project_root}")

os.chdir(_project_root)
for _p in (_project_root, _project_root / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from gradient_ascent.experiments_notebook import experiments_bootstrap

_ctx = experiments_bootstrap()
DEFAULT_OUT_DIR = _ctx.default_out_dir
OUT_DIR = DEFAULT_OUT_DIR
wandb = _ctx.wandb
IN_COLAB = _ctx.in_colab


## Section 1 - Core training and unlearning checkpoints

This section trains the original and retrained models, then runs five unlearning baselines: gradient ascent, SSD, SalUn, certified removal, and SCRUB. For GA, the preset below stays faithful to vanilla gradient ascent on the forget set, but is made slightly stronger in an easy-to-justify way: a modestly higher learning rate, more forget-set updates per epoch, BatchNorm buffers still frozen for clean evaluation, and a looser clip so updates are visible without becoming unstable.

It also saves simple baseline-specific diagnostics that are easy to explain in a dissertation. For GA, these are the mean forget-set cross-entropy and gradient norm at each unlearning step. For SCRUB, they are the student-teacher KL divergence on the forget and retain sets, the retain-set cross-entropy, and retain/forget accuracy. Together these show whether SCRUB is separating from the teacher on forgotten data while still staying close on retained data.

**Configuration:** edit `DEFAULT_CORE_CONFIG` in `gradient_ascent.experiments_notebook` (or assign a fresh `CoreSectionConfig`) before running the code cell if you need different reuse flags or diagnostics.

In [ ]:
from gradient_ascent.experiments_notebook import DEFAULT_CORE_CONFIG, experiments_section_core

# Optional: edit DEFAULT_CORE_CONFIG (reuse flags, diagnostics) before running.
runtime, core_artifacts, core_cfg = experiments_section_core(
    out_dir=OUT_DIR,
    wandb_module=wandb,
    config=DEFAULT_CORE_CONFIG,
)


## Section 2 - Trajectories, MIA, and evolving similarity bar plots

Initialises shared similarity helpers, then runs snapshot trajectories for all five baselines (GA, SSD, SalUn, certified removal, and SCRUB), computes MIA trajectories, and generates PDF-friendly similarity summaries plus two evolving bar visualisations for both references: (1) mean-across-layers bars and (2) grouped bars by metric with per-layer bars.

In [ ]:
from gradient_ascent.experiments_notebook import experiments_section_trajectory

similarity_setup, trajectory_artifacts, trajectory_wandb_run = experiments_section_trajectory(
    runtime,
    core_artifacts,
    wandb_module=wandb,
)


## Section 3 - Combined cross-algorithm comparison

Overlays all five baselines, including SCRUB, in a single consolidated similarity + MIA comparison figure.

In [ ]:
from gradient_ascent.experiments_notebook import experiments_section_combined

combined_path = experiments_section_combined(runtime, wandb_module=wandb)


## Section 4 - Multi-target averaging across all forget classes

Runs the full pipeline for each forget label (`0..9`) and then aggregates outputs:

- Utility plots become two-series curves: `forgotten class` and `retained classes (mean)`.
- Similarity trajectories are averaged over forget labels for each algorithm/reference pair.

This section is compute-heavy because it effectively multiplies core + trajectory work by 10.

The code cell below optionally seeds `target_6` from an earlier single-target run (frog), then runs `run_multitarget_averaged_experiment` for all forget labels and builds aggregate artefacts. Toggle `DEFAULT_MULTITARGET_CONFIG` in `experiments_notebook` (or pass a custom `MultitargetSectionConfig`) for `run_similarity_stage` and related options.

In [ ]:
from gradient_ascent.experiments_notebook import (
    DEFAULT_MULTITARGET_CONFIG,
    experiments_section_multitarget,
)

# Optional: edit DEFAULT_MULTITARGET_CONFIG (e.g. run_similarity_stage) before running.
multi_target_artifacts = experiments_section_multitarget(
    runtime,
    out_dir=OUT_DIR,
    wandb_module=wandb,
    core_config=core_cfg,
    multitarget_config=DEFAULT_MULTITARGET_CONFIG,
)


## Section 5 — Correlation: similarity as a proxy for MIA and accuracy

After **Section 4** has produced per-target CSVs under `multitarget_aggregate/`, this section asks whether **representation similarity** (to retrain or original) tracks **forget-set MIA** and **utility** as unlearning progresses.

1. **Run-level (endpoint change):** **signed** scalar similarity movement (layer with largest endpoint change, unified “positive = more similar” convention) vs **MIA reduction** and vs **forget-class accuracy reduction** (start − end on the forgotten class) — scatter plots and pooled correlations under `correlation/`. Absolute-magnitude variants of the same deltas are not plotted (they duplicate information when summarising monotonic relationships).
2. **Step-wise (as unlearning takes place):** at each unlearning epoch, **Spearman r** across all `(algorithm × forget class)` runs between **layer-mean oriented similarity** (each of the five metrics) and three outcomes: **MIA** (`forget_logreg_mean_member_prob`), **forget-class accuracy**, and **mean retained-class accuracy** — saved under `correlation/epochwise/` with line plots of *r* vs step.

Interpretation is exploratory: about 50 cross-sectional points per epoch; see `docs/evaluation/similarity-mia-correlation.md`.

In [ ]:
from gradient_ascent.experiments_notebook import experiments_section_correlation

_correlation_paths = experiments_section_correlation(runtime, out_dir=OUT_DIR)
